# Run 2D Sweep 

In [ ]:
"""
2D Parameter Sweep: Amplitude and K_c (Duffing nonlinearity) using Finite Element
Sweeps over both excitation amplitude and cubic stiffness coefficient
Runs simulations in parallel using joblib
Similar to the ROM version but using FE discretization
"""

import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from joblib import Parallel, delayed
import itertools
import traceback
import json
from datetime import datetime

# Add project root to path
project_root = Path(__file__).resolve().parents[3]
sys.path.append(str(project_root))

from Modeling.models.beam_properties import PiezoBeamParams
from Modeling.models.FE1 import PiezoBeamFE, solve_newmark

# ======= Sweep Parameters =======
K_i = 1800
K_p = 0.018
R_c = 1e3
t_end = 1.0
f0 = 1000
f1 = 3000
dt = 1/f1/50
npz_filename = f'FE_2D_sweep_hardening_Kp={K_p:.3f}_Ki={K_i:.0f}.npz'
# Parameter lists
amp_list = np.array([0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]) * 125  # Amplitude sweep
# Kc_list = -np.array([ 4e9, 8e9, 1e10, 1.2e10, 1.3e10, 1.4e10, 1.5e10, 1.6e10])  # K_c (Duffing) sweep
Kc_list = np.array([  8e9, 1.2e10, 1.5e10, 1.8e10, 2.25e10, 2.5e10, 2.75e10, 3e10])  # K_c (Duffing) sweep

print(f"Amplitude sweep: {amp_list}")
print(f"K_c sweep: {Kc_list}")
print(f"Total simulations: {len(amp_list) * len(Kc_list)}")

# Setup FE model
params_fe = PiezoBeamParams(hp=0.252e-3, hs=0.51e-3
                            , d31= -1.48e-10,eps_r=1700
                            )
params_fe.zeta_p = 0.0151 * 8
params_fe.zeta_q = 0.0392 * 10
fe = PiezoBeamFE(params_fe)

# ======= Function to run single simulation =======
def run_single_simulation_2d_fe(amp, kc, K_p, K_i, R_c, fe_params, dt, t_end, f0, f1):
	"""Run a single FE simulation for given amplitude and K_c parameters.
	
	Returns:
		{"status": "ok", "result": {...}} on success
		{"status": "failed", "error": error_message, "amp": amp, "kc": kc} on failure
	"""
	try:
		print(f"  Amp = {amp:.1f} V, K_c = {kc:.2e}")
		
		# Recreate FE object in each worker (for parallel safety)
		fe_local = PiezoBeamFE(fe_params, n_el_gap=1, n_el_patch=3)
		
		# Create excitation function for this amplitude
		def v_exc(t_var, A_exc=amp, f0=f0, f1=f1, t_end=t_end):
			return A_exc * np.sin(2*np.pi*(f0 + t_var*(f1-f0)/t_end) * t_var)
		
		# Build ODE system
		ode = fe_local.build_ode_system(
			j_exc=30,
			K_c=kc,
			K_i=K_i,
			K_p=K_p,
			R_c=R_c,
			v_exc=v_exc
		)
		
		# Run time-domain simulation
		result = solve_newmark(
			ode=ode,
			dt=dt,
			t_end=t_end,
			beta=0.25,
			gamma=0.5,
			newton_tol=1e-8,
			newton_maxiter=8,
			x0=np.zeros(ode.M.shape[0]),
			x_dot0=np.zeros(ode.M.shape[0]),
			do_spectral=True
		)
		
		return {
			"status": "ok",
			"result": {
				"amp": amp,
				"kc": kc,
				"t": result['t'],
				"u": result['u'],
				"u_dot": result['u_dot'],
				"u_ddot": result['u_ddot'],
				"q": result['q'],
				"v": result['v'],
				"freq": result['spectral']['freq'],
				"FRF": result['spectral']['FRF'],
				"spectrum": result['spectral']
			}
		}
	
	except Exception as e:
		print(f"\n{'='*70}")
		print(f"ERROR: Simulation failed for Amp={amp:.1f}V, K_c={kc:.2e}")
		print(f"Exception: {type(e).__name__}: {str(e)}")
		print(f"{'='*70}\n")
		
		return {
			"status": "failed",
			"amp": amp,
			"kc": kc,
			"error": str(e),
			"exception_type": type(e).__name__
		}

# ======= Run 2D sweep in parallel =======
print("Running 2D parameter sweep in parallel...")
print(f"Note: Errors in individual simulations will NOT stop the sweep.")
print(f"Partial results will be saved for successfully completed simulations.\n")

param_pairs = list(itertools.product(amp_list, Kc_list))

all_results = Parallel(n_jobs=32, verbose=10)(
	delayed(run_single_simulation_2d_fe)(amp, kc, K_p, K_i, R_c, params_fe, dt, t_end, f0, f1) 
	for amp, kc in param_pairs
)

# ======= Separate successful and failed simulations =======
sim_results = []
failed_sims = []

for res in all_results:
	if res["status"] == "ok":
		sim_results.append(res["result"])
	else:
		failed_sims.append({
			"amp": res["amp"],
			"kc": res["kc"],
			"exception": res.get("exception_type", "Unknown"),
			"message": res["error"]
		})

print(f"\n{'='*70}")
print(f"SWEEP STATUS:")
print(f"  Total simulations: {len(all_results)}")
print(f"  Successful: {len(sim_results)}")
print(f"  Failed: {len(failed_sims)}")
print(f"{'='*70}\n")

if failed_sims:
	print("FAILED SIMULATIONS:")
	print("-" * 70)
	for fail in failed_sims:
		print(f"  Amp={fail['amp']:.1f}V, K_c={fail['kc']:.2e}: {fail['exception']}")
		print(f"    → {fail['message']}")
	print("-" * 70 + "\n")

# ======= Organize results by K_c =======
results_by_Kc = {}
common_freq = None

for res in sim_results:
	kc = res["kc"]
	if kc not in results_by_Kc:
		results_by_Kc[kc] = {
			"amps": [],
			"FRFs": [],
			"time_histories": [],
			"freq": None,
			"t": None
		}
	
	results_by_Kc[kc]["amps"].append(res["amp"])
	results_by_Kc[kc]["FRFs"].append(res["FRF"])
	
	# Store complete time domain output
	results_by_Kc[kc]["time_histories"].append({
		"amp": res["amp"],
		"t": res["t"],
		"u": res["u"],
		"u_dot": res["u_dot"],
		"u_ddot": res["u_ddot"],
		"q": res["q"],
		"v": res["v"]
	})
	
	if results_by_Kc[kc]["freq"] is None:
		results_by_Kc[kc]["freq"] = res["freq"]
		results_by_Kc[kc]["t"] = res["t"]
	
	if common_freq is None:
		common_freq = res["freq"]

# ======= Create subplots for each K_c value =======
fig, axes = plt.subplots(len(Kc_list), 1, figsize=(10, 4*len(Kc_list)))
if len(Kc_list) == 1:
	axes = [axes]

for idx, kc in enumerate(sorted(results_by_Kc.keys())):
	ax = axes[idx]
	data = results_by_Kc[kc]
	
	cmap = plt.cm.viridis
	colors = cmap(np.linspace(0, 1, len(data["amps"])))
	
	for i, (amp, FRF) in enumerate(zip(data["amps"], data["FRFs"])):
		ax.semilogy(data["freq"], FRF, '.', color=colors[i], linewidth=2, 
		           label=f"A = {amp:.1f} V")
	
	ax.set_xlim([1300, 3000])
	ax.set_ylim([3e-5, 6e-4])
	ax.grid(True)
	ax.legend(loc='best', fontsize=8)
	ax.set_xlabel("Frequency [Hz]")
	ax.set_ylabel("FRF Magnitude")
	ax.set_title(f"FE: Amplitude Sweep at K_c = {kc:.2e}")

plt.tight_layout()

# Create sim_dat directory if it doesn't exist
sim_dat_dir = Path(__file__).parent / 'sim_dat'
sim_dat_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(sim_dat_dir / f'FE_2D_sweep_amp_Kc_Kp={K_p:.3f}_Ki={K_i:.0f}.png', dpi=300, bbox_inches='tight')
print(f"Figure saved to: {sim_dat_dir / f'FE_2D_sweep_amp_Kc_Kp={K_p:.3f}_Ki={K_i:.0f}.png'}")
plt.close(fig)

# ======= Save results =======

np.savez(sim_dat_dir / npz_filename, 
         amp_list=amp_list, 
         Kc_list=Kc_list,
         K_p=K_p,
         K_i=K_i,
         R_c=R_c,
         t_end=t_end,
         f0=f0,
         f1=f1,
         dt=dt,
         # FE model parameters
         params_fe_hp=params_fe.hp,
         params_fe_hs=params_fe.hs,
         params_fe_d31=params_fe.d31,
         params_fe_eps_r=params_fe.eps_r,
         params_fe_zeta_p=params_fe.zeta_p,
         params_fe_zeta_q=params_fe.zeta_q,
         results_by_Kc=results_by_Kc)
print(f"Results saved to: {sim_dat_dir / npz_filename}")

# ======= Save error log if there were failures =======
if failed_sims:
	error_log = {
		"timestamp": datetime.now().isoformat(),
		"total_simulations": len(all_results),
		"successful_simulations": len(sim_results),
		"failed_simulations": len(failed_sims),
		"failed_cases": failed_sims,
		"sweep_parameters": {
			"K_p": K_p,
			"K_i": K_i,
			"R_c": R_c,
			"t_end": t_end,
			"f0": f0,
			"f1": f1,
			"dt": dt
		}
	}
	
	error_log_path = sim_dat_dir / f'FE_2D_sweep_errors_Kp={K_p:.3f}_Ki={K_i:.0f}.json'
	with open(error_log_path, 'w') as f:
		json.dump(error_log, f, indent=2)
	print(f"Error log saved to: {error_log_path}")

print("\n" + "="*70)
print("2D FE SWEEP COMPLETE")
print("="*70)
print(f"Amplitude range: {amp_list[0]:.1f} - {amp_list[-1]:.1f} V")
print(f"K_c range: {Kc_list[0]:.2e} - {Kc_list[-1]:.2e}")
print(f"K_p: {K_p}, K_i: {K_i}")
print(f"Total simulations attempted: {len(all_results)}")
print(f"Successful simulations: {len(sim_results)}")
if failed_sims:
	print(f"Failed simulations: {len(failed_sims)}")
	print(f"Success rate: {100*len(sim_results)/len(all_results):.1f}%")
print("\nOUTPUTS SAVED:")
print(f"  - NPZ file with all results: {npz_filename}")
print(f"  - Visualization: FE_2D_sweep_amp_Kc_*.png")
if failed_sims:
	print(f"  - Error log: FE_2D_sweep_errors_*.json")
print("="*70)


# Load 2D sweep results

In [22]:
type(data['K_i'] ) is np.ndarray
float(data['K_i'])

2000.0

In [23]:
"""
Plot amplitude sweeps for different K_c values in separate figures (FE version)
Similar to the ROM plotting script but for Finite Element results
Includes comparison with ROM and reference data
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import sys
from pathlib import Path
import os

# Add project root to path
# project_root = Path(__file__).resolve().parents[3]
project_root = Path.cwd().parents[2]
sys.path.append(str(project_root))

from Modeling.models.beam_properties import PiezoBeamParams
from Modeling.models.FE1 import PiezoBeamFE, frf_sweep
scalar = 1/1.6  # Scale factor for FRF plotting
# ======= Load FE 2D Sweep Data =======
script_dir = project_root = Path.cwd()
# npz_path_main = script_dir / 'sim_dat' / 'FE_2D_sweep_softening_Kp=0.02_Ki=1800.npz'
npz_path_main = Path(r"C:\Users\setemadi3\GaTech Dropbox\Seyednima Etemadi\Projects\Metamaterial beam\Modeling\tasks\FE_studies\sim_dat\FE_2D_sweep_softening_Kp=0.030_Ki=2000.npz")
# ======= Create Output Directory =======
if not npz_path_main.exists():
    # Try alternative naming
    sim_dat_dir = script_dir / 'sim_dat'
    if sim_dat_dir.exists():
        npz_files = list(sim_dat_dir.glob('FE_2D_sweep_*.npz'))
        if npz_files:
            npz_path_main = npz_files[0]
            print(f"Found NPZ file: {npz_path_main.name}")
        else:
            print(f"No FE 2D sweep data found in {sim_dat_dir}")
            print("Please run FE_2D_sweep_amp_Kc.py first")
            exit(1)
    else:
        print(f"Directory {sim_dat_dir} not found")
        exit(1)

data = np.load(npz_path_main, allow_pickle=True)

amp_list = data['amp_list']
Kc_list = data['Kc_list']
results_by_Kc = data['results_by_Kc'].item()  # Dictionary from NPZ (keys may be tuple/np scalar/str)

def _kc_to_float(k):
    """Convert K_c keys (including tuple/list/array wrappers) to scalar float."""
    if isinstance(k, (tuple, list)):
        if len(k) == 0:
            raise ValueError('Empty K_c key container')
        k = k[0]
    if isinstance(k, np.ndarray):
        if k.size == 0:
            raise ValueError('Empty K_c ndarray key')
        k = k.reshape(-1)[0]
    return float(k)

# Map numeric K_c value -> original dictionary key for robust lookup
kc_key_map = {}
unparsed_keys = []
for k in results_by_Kc.keys():
    try:
        kc_key_map[_kc_to_float(k)] = k
    except (TypeError, ValueError) as e:
        unparsed_keys.append((k, str(e)))

if not kc_key_map:
    raise ValueError('Could not parse any K_c keys from results_by_Kc')

print(f"Loaded FE data from: {npz_path_main.name}")
print(f"  {len(Kc_list)} K_c values, {len(amp_list)} amplitudes")
print(f"  K_c values: {Kc_list}")
print(f"  Amplitude values: {amp_list}")

# Sort K_c values based on available parsed keys (prevents KeyError/type issues)
kc_order = np.sort(np.array(list(kc_key_map.keys()), dtype=float))

kc_from_file = np.sort(np.array(Kc_list, dtype=float))
if kc_from_file.shape != kc_order.shape or not np.allclose(kc_from_file, kc_order):
    print("  Warning: Kc_list in NPZ does not match parsed results_by_Kc keys; using keys from results_by_Kc.")
if unparsed_keys:
    print(f"  Note: skipped {len(unparsed_keys)} unparsed K_c key(s).")

# Get sample frequency array from first result
sample_kc = kc_key_map[kc_order[0]]
freq = results_by_Kc[sample_kc]['freq']
print(f"  Frequency range: {freq[0]:.1f} - {freq[-1]:.1f} Hz ({len(freq)} points)")


Loaded FE data from: FE_2D_sweep_softening_Kp=0.030_Ki=2000.npz
  9 K_c values, 7 amplitudes
  K_c values: [-2.00e+09 -4.00e+09 -8.00e+09 -1.00e+10 -1.20e+10 -1.60e+10 -2.00e+10
 -2.25e+10 -2.50e+10]
  Amplitude values: [ 6.25 12.5  18.75 25.   31.25 37.5  50.  ]
  Frequency range: 0.0 - 74999.5 Hz (75001 points)

Computing FE reference responses (frequency domain)...


FRF sweep: 100%|██████████| 500/500 [00:03<00:00, 154.82it/s]


  FE linear (K_c=0, K_p=0.03)


FRF sweep: 100%|██████████| 500/500 [00:03<00:00, 166.10it/s]


  FE short circuit (K_c=0, K_p=100)
  Experimental linear (K_p=0.03)
  Experimental short circuit (K_p=100)

Generating plots for each K_c value...


WindowsPath('C:/Users/setemadi3/GaTech Dropbox/Seyednima Etemadi/Projects/Metamaterial beam/Modeling/tasks/FE_studies/sim_dat/FE_2D_sweep_softening_Kp=0.030_Ki=2000.npz')

In [39]:

# ======= FE Reference Data (Frequency Domain) =======
print("\nComputing FE reference responses (frequency domain)...")
K_p = data['K_p']
K_i = 1900#float(data['K_i'])
params_fe = PiezoBeamParams(
#                             hp=0.252e-3, hs=0.51e-3
# #                             , d31= -1.48e-10,eps_r=1700
                            )
params_fe.zeta_p = 0.0151 * 8
params_fe.zeta_q = 0.0392 * 10

fe = PiezoBeamFE(params_fe)

# Linear case (K_c = 0)
def v_exc_ref(t, A_exc=1.0):
    return A_exc

ode_linear = fe.build_ode_system(
    j_exc=30,
    K_c=0,
    K_i=K_i,
    K_p=K_p,
    R_c=1e3,
    v_exc=v_exc_ref
)

f_fe_ref = np.linspace(1000, 3000, 500)
frf_linear_fd = frf_sweep(ode_linear, f_fe_ref * 2 * np.pi)
FRF_FE_linear = np.mean(np.abs(frf_linear_fd['u_dot']), axis=1)

ode_linear_lowdamping = fe.build_ode_system(
    j_exc=30,
    K_c=0,
    K_i=K_i,
    K_p=0.004,
    R_c=1e3,
    v_exc=v_exc_ref
)
frf_linear_lowdamping_fd = frf_sweep(ode_linear_lowdamping, f_fe_ref * 2 * np.pi)
FRF_FE_linear_lowdamping = np.mean(np.abs(frf_linear_lowdamping_fd['u_dot']), axis=1)

print(f"  FE linear (K_c=0, K_p={K_p})")

# Short Circuit (very high K_p)
ode_sc = fe.build_ode_system(
    j_exc=30,
    K_c=0,
    K_i=K_i,
    K_p=100,
    R_c=1e3,
    v_exc=v_exc_ref
)

frf_sc_fd = frf_sweep(ode_sc, f_fe_ref * 2 * np.pi)
FRF_FE_SC = np.mean(np.abs(frf_sc_fd['u_dot']), axis=1)

print(f"  FE short circuit (K_c=0, K_p=100)")

# ======= Experimental Reference Data (if available) =======
try:
    # Load experimental data from Z drive
    npz_path_linear = r"Z:\Nima\Synthetic_impedance\long_beam\ssdsl_dat\ssdsl_dat_Nov\11\linear_lowdamping.npz"

    npz_path_SC = r"Z:\Nima\Synthetic_impedance\long_beam\ssdsl_dat\ssdsl_dat_Nov\7\kc0_kp_sweep\OCSC\SC.npz"
    
    data_linear = np.load(npz_path_linear)
    data_SC = np.load(npz_path_SC)
    
    freq_exp_linear = data_linear['freq']
    frf_exp_linear = np.mean(data_linear['frf_data'], axis=1)  # Average over measurement points
    
    freq_exp_SC = data_SC['freq']
    frf_exp_SC = np.mean(data_SC['frf_mag'], axis=1)  # Average over measurement points
    
    have_exp = True
    print(f"  Experimental linear (K_p={K_p})")
    print(f"  Experimental short circuit (K_p=100)")
except Exception as e:
    have_exp = False
    print(f"  Experimental data not available: {e}")



# ======= Plot for Each K_c Value =======
print(f"\nGenerating plots for each K_c value...")

# Collect all unique amplitudes across all K_c values for consistent coloring
all_amps = []
for kc in kc_order:
    kc_key = kc_key_map[kc]
    all_amps.extend(results_by_Kc[kc_key]["amps"])
unique_amps = np.sort(np.unique(all_amps))

fig_path = script_dir / 'sim_dat' / 'Softening_Feb22-2026'
fig_path.mkdir(parents=True, exist_ok=True)
# Create color mapping
cmap = cm.get_cmap("viridis", len(unique_amps))
amp_to_color = {amp: cmap(i / (len(unique_amps) - 1)) for i, amp in enumerate(unique_amps)}

for idx, kc in enumerate(kc_order):
    kc_key = kc_key_map[kc]
    data_kc = results_by_Kc[kc_key]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot amplitude sweeps
    for j, amp in enumerate(data_kc["amps"]):
        color = amp_to_color[amp]
        FRF = data_kc["FRFs"][j]
        ax.semilogy(
            data_kc["freq"], FRF * scalar,
            '.-',
            color=color,
            linewidth=2,
            markersize=4,
            label=f"Amp={amp:.1f}V"
        )
    
    # Add FE reference lines
    ax.semilogy(f_fe_ref, FRF_FE_linear * scalar, 'k--', linewidth=2.5, 
                label=f'FE linear (K_p={K_p})', alpha=0.8)
    # ax.semilogy(f_fe_ref, FRF_FE_SC * scalar, '--', color='purple', linewidth=2.5, 
    #             label='FE short circuit (K_p=100)', alpha=0.8)
    ax.semilogy(f_fe_ref, FRF_FE_linear_lowdamping * scalar, ':', linewidth=2.0,
                label=f'FE linear low damping (K_p=0.015)', alpha=0.6)
    
    # Add experimental reference if available
    if have_exp:
        ax.semilogy(freq_exp_linear, frf_exp_linear, 'r--', linewidth=2.5, 
                    label=f'Exp. linear (K_p={K_p})', alpha=0.7)
        # ax.semilogy(freq_exp_SC, frf_exp_SC, ':', color='darkviolet', linewidth=2.5, 
        #             label='Exp. short circuit (K_p=100)', alpha=0.7)
    
    ax.set_xlabel("Frequency [Hz]", fontsize=12)
    ax.set_ylabel('FRF Magnitude (m/s/V)', fontsize=12)
    ax.set_title(f"FE Amplitude Sweep: K_c = {kc:.2e}", fontsize=13, fontweight='bold')
    ax.grid(True, which='both', linestyle='--', alpha=0.4)
    # ax.legend(fontsize=9, loc='best', framealpha=0.95)
    ax.set_xlim([1200, 3000])
    ax.set_ylim([1e-5, 1e-3])
    
    plt.tight_layout()
    
    # Save figure
    save_path = fig_path / f"FE_kc_{kc:.2e}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"  Saved: {save_path.name}")
    plt.close(fig)
    plt.show(block=False)

print(f"\nAll amplitude sweep plots saved to: {fig_path}/")

# ======= Create Legend Figure =======
print("Creating legend figure...")

amp_handles = [mpatches.Patch(color=amp_to_color[a], label=f"Amp={a:.1f}V") 
               for a in unique_amps]

ref_handles = [
    Line2D([], [], color='red', linestyle='--', linewidth=2.5, label=f'FE linear (K_p={K_p})'),
    Line2D([], [], color='purple', linestyle='--', linewidth=2.5, label='FE short circuit (K_p=100)'),
]

if have_exp:
    ref_handles.extend([
        Line2D([], [], color='darkred', linestyle=':', linewidth=2.5, label=f'Exp. linear (K_p={K_p})'),
        Line2D([], [], color='darkviolet', linestyle=':', linewidth=2.5, label='Exp. short circuit (K_p=100)'),
    ])

fig_leg = plt.figure(figsize=(10, 8))
fig_leg.legend(handles=amp_handles + ref_handles, loc='center', ncol=2, fontsize=11, 
               frameon=True, title='FE 2D Amplitude & K_c Sweep', title_fontsize=12)
plt.axis('off')
plt.tight_layout()

legend_path = fig_path / 'FE_legend_amp_sweep_by_Kc.png'
plt.savefig(legend_path, dpi=300, bbox_inches='tight')
print(f"Saved legend: {legend_path.name}")
plt.close(fig_leg)

# ======= Summary Statistics =======
print("\n" + "="*70)
print("FE AMPLITUDE SWEEP PLOT SUMMARY")
print("="*70)
print(f"Number of K_c values: {len(kc_order)}")
print(f"Number of amplitude values: {len(unique_amps)}")
print(f"Total plots generated: {len(kc_order)}")
print(f"Output directory: {fig_path}/")
print("="*70)

# ======= Optional: Comparison Table =======
print("\nFrequency shift analysis (peak frequency vs amplitude):")
print("-" * 70)
for kc in kc_order[:3]:  # Show first 3 K_c values as example
    kc_key = kc_key_map[kc]
    data_kc = results_by_Kc[kc_key]
    print(f"\nK_c = {kc:.2e}:")
    for j, amp in enumerate(data_kc["amps"][:3]):  # First 3 amplitudes
        FRF = data_kc["FRFs"][j]
        freq_local = data_kc["freq"]
        # Find peak frequency (in frequency range of interest)
        mask = (freq_local >= 1200) & (freq_local <= 3000)
        if np.any(mask):
            peak_idx = np.argmax(FRF[mask])
            peak_freq = freq_local[np.where(mask)[0][peak_idx]]
            peak_val = FRF[np.where(mask)[0][peak_idx]]
            print(f"  Amp={amp:.1f}V:  peak at {peak_freq:.0f} Hz, magnitude={peak_val:.2e}")

print("\n" + "="*70)



Computing FE reference responses (frequency domain)...


FRF sweep: 100%|██████████| 500/500 [00:02<00:00, 226.51it/s]


  FE linear (K_c=0, K_p=0.03)


FRF sweep: 100%|██████████| 500/500 [00:02<00:00, 223.76it/s]
C:\Users\setemadi3\AppData\Local\Temp\ipykernel_16896\2176394644.py:97: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("viridis", len(unique_amps))


  FE short circuit (K_c=0, K_p=100)
  Experimental linear (K_p=0.03)
  Experimental short circuit (K_p=100)

Generating plots for each K_c value...
  Saved: FE_kc_-2.50e+10.png
  Saved: FE_kc_-2.25e+10.png
  Saved: FE_kc_-2.00e+10.png
  Saved: FE_kc_-1.60e+10.png
  Saved: FE_kc_-1.20e+10.png
  Saved: FE_kc_-1.00e+10.png
  Saved: FE_kc_-8.00e+09.png
  Saved: FE_kc_-4.00e+09.png
  Saved: FE_kc_-2.00e+09.png

All amplitude sweep plots saved to: c:\Users\setemadi3\GaTech Dropbox\Seyednima Etemadi\Projects\Metamaterial beam\Modeling\tasks\FE_studies\sim_dat\Softening_Feb22-2026/
Creating legend figure...
Saved legend: FE_legend_amp_sweep_by_Kc.png

FE AMPLITUDE SWEEP PLOT SUMMARY
Number of K_c values: 9
Number of amplitude values: 7
Total plots generated: 9
Output directory: c:\Users\setemadi3\GaTech Dropbox\Seyednima Etemadi\Projects\Metamaterial beam\Modeling\tasks\FE_studies\sim_dat\Softening_Feb22-2026/

Frequency shift analysis (peak frequency vs amplitude):
---------------------------